# SO-app v0.1 en Kaggle — SO como app (userland nativo con proot)

Corre un Ubuntu 22.04 **nativo** sobre el kernel del host con `proot`: Chrome real + Node + OpenCode adentro, arranque en ~0.014s (vs 10-15 min de QEMU/TCG, único emulador viable en Kaggle sin KVM).

- **Celda 1**: instala `proot`, baja `ubuntu-base` (29MB), fija DNS/apt, instala `curl`.
- **Celda 2**: instala Chrome stable Linux y verifica Node/OpenCode por bind.
- **Celda 3**: verificación final + tiempos. Todo idempotente: re-correr reusa lo existente.
- Tiempo total primera vez: ~5-8 min. Siguientes: segundos.

In [ ]:
%%bash
set -u
DIR=/kaggle/working/vmdata/proot
mkdir -p "$DIR"
command -v proot >/dev/null || { apt-get update -qq && apt-get install -y -qq proot; }
proot --version | head -2
if [ ! -d "$DIR/rootfs/bin" ]; then
  echo "== descargando ubuntu-base 22.04 (~29MB) =="
  curl -sSL --retry 3 -o "$DIR/ubuntu-base.tar.gz" \
    https://cdimage.ubuntu.com/ubuntu-base/releases/22.04/release/ubuntu-base-22.04-base-amd64.tar.gz
  mkdir -p "$DIR/rootfs"
  tar -xzf "$DIR/ubuntu-base.tar.gz" -C "$DIR/rootfs"
else
  echo "== rootfs existente, reuso =="
fi
# DNS publico: el de Docker (127.0.0.11) cuelga apt dentro de proot
printf 'nameserver 8.8.8.8\nnameserver 1.1.1.1\n' > "$DIR/rootfs/etc/resolv.conf"
echo "== boot test =="
time proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -w / /bin/sh -c 'echo PROOT_OK $(grep PRETTY_NAME /etc/os-release)'
echo "== apt + curl =="
proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -w / /bin/sh -c \
  'apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true update && apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true install -y curl ca-certificates' | tail -2

In [ ]:
%%bash
set -u
DIR=/kaggle/working/vmdata/proot
R() { proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -w / "$@"; }
if [ ! -x "$DIR/rootfs/opt/google/chrome/chrome" ]; then
  echo "== descargando Chrome (~137MB) =="
  R /bin/sh -c 'curl -sSL --retry 2 -o /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && ls -lh /tmp/chrome.deb'
  echo "== instalando (errores dpkg de iconos/systemd se ignoran: el binario corre igual) =="
  R /bin/sh -c 'apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true install -y /tmp/chrome.deb || true; rm -f /tmp/chrome.deb; apt-get -o APT::Sandbox::User=root clean; rm -rf /var/lib/apt/lists/*' | tail -2
else
  echo "== Chrome existente, reuso =="
fi
echo "== Chrome =="; R /bin/sh -c 'LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu /opt/google/chrome/chrome --version --no-sandbox'
echo "== Node (bind host) =="; proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -b /tools -w / /tools/node/bin/node --version
echo "== OpenCode (bind host) =="; proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -b /tools -w / /tools/node/bin/opencode --version

In [ ]:
%%bash
DIR=/kaggle/working/vmdata/proot
R() { proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -w / "$@"; }
echo "== headless render =="
R /opt/google/chrome/chrome --headless --no-sandbox --disable-gpu --dump-dom about:blank 2>/dev/null
echo "== tamano =="; du -sh "$DIR/rootfs"
echo OK